In [38]:
# ======================================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ======================================================================
# These are the tools we'll use to load, clean, and transform our data

import json                              # For saving metadata
from pathlib import Path

import joblib                           # For saving the preprocessor
import numpy as np                       # For math operations
import pandas as pd                      # For working with data tables
from sklearn.compose import ColumnTransformer  # To apply different transformations to different columns
from sklearn.impute import SimpleImputer      # To handle missing values
from sklearn.pipeline import Pipeline         # To chain transformations together
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # To encode and scale features

In [39]:
# ======================================================================
# STEP 2: LOAD RAW DATA
# ======================================================================
# Set up folder structure and load the original CSV file

DATA_VERSION = "v2"  # Version number for our processed data
RAW_DATA_PATH = Path("online_shoppers_intention.csv")  # Where the original data is
ARTIFACT_ROOT = Path("artifacts")  # Where we'll save processed data
DATA_ARTIFACT_DIR = ARTIFACT_ROOT / DATA_VERSION  # Folder for this version
DATA_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)  # Create the folder

TARGET_COL = "Revenue"  # This is what we want to predict (buy or not)

# Read the CSV file into a table (DataFrame)
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"✓ Loaded raw data: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()  # Show first few rows

✓ Loaded raw data: 12330 rows, 18 columns


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [40]:
# ======================================================================
# STEP 3: VALIDATE DATA (Check that all expected columns exist)
# ======================================================================
# Make sure our raw data has all the features we need

# List of all columns we expect from the specification
expected_columns = [
    "Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates",
    "PageValues", "SpecialDay", "Month", "OperatingSystems", "Browser",
    "Region", "TrafficType", "VisitorType", "Weekend", TARGET_COL
]

# Check if any columns are missing
missing_columns = sorted(set(expected_columns) - set(df_raw.columns))
if missing_columns:
    raise ValueError(f"❌ ERROR: Missing columns: {missing_columns}")

# Create a clean copy with only the columns we need
df = df_raw[expected_columns].copy()

# Count problems in the data
duplicate_count = int(df.duplicated().sum())
missing_counts = df.isna().sum().sort_values(ascending=False)
target_dist = df[TARGET_COL].value_counts(dropna=False)

# Print data quality summary
print("="*60)
print("DATA QUALITY CHECK")
print("="*60)
print(f"All {len(expected_columns)} required columns are present")
print(f"\n📊 Current data shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"📊 Duplicate rows found: {duplicate_count}")
print(f"\n❓ Missing values per column:")
print(missing_counts[missing_counts > 0] if (missing_counts > 0).any() else "  None - data is complete!")
print(f"\n📈 Target distribution (Revenue):")
print(target_dist)
print("="*60)

DATA QUALITY CHECK
All 18 required columns are present

📊 Current data shape: 12330 rows × 18 columns
📊 Duplicate rows found: 125

❓ Missing values per column:
  None - data is complete!

📈 Target distribution (Revenue):
Revenue
False    10422
True      1908
Name: count, dtype: int64


In [41]:
# ======================================================================
# STEP 4: CLEAN & ENGINEER FEATURES
# ======================================================================
# Fix data issues and create new useful features from existing ones

print("\n" + "="*60)
print("CLEANING DATA...")
print("="*60)

# 4.1) CHECK FOR DUPLICATES (but keep them - they're real sessions!)
print("\n1️⃣ Checking for duplicate rows...", end=" ")
duplicate_count = df.duplicated().sum()
print(f"Found {duplicate_count} duplicate rows")
if duplicate_count > 0:
    print(f"   ℹ️  Note: These represent real visitor sessions with identical patterns.")
    print(f"   They are KEPT (not removed) because each row = one session.")

# 4.2) Convert YES/NO to 1/0 (True/False to 1/0)
print("\n2️⃣ Converting Weekend & Revenue to 0/1 format...", end=" ")
bool_map = {"TRUE": 1, "FALSE": 0, "True": 1, "False": 0, True: 1, False: 0, 1: 1, 0: 0}
for col in ["Weekend", TARGET_COL]:
    df[col] = df[col].map(bool_map)
print("Done!")

# 4.3) Convert month names to numbers (for cyclic features)
print("3️⃣ Converting month names to numbers...", end=" ")
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
month_to_num = {m: i + 1 for i, m in enumerate(month_order)}
df["Month_num"] = df["Month"].map(month_to_num)
if df["Month_num"].isna().any():
    df["Month_num"] = df["Month_num"].fillna(df["Month_num"].mode().iloc[0])
print("Done!")

# 4.4) CREATE NEW FEATURES (from your project spec)
print("\n4️⃣ Creating engineered features...")

# Feature 1: Total time spent on all pages
print("   - TotalSessionDuration (sum of all page durations)", end=" ")
df["TotalSessionDuration"] = (
    df["Administrative_Duration"] + df["Informational_Duration"] + df["ProductRelated_Duration"]
)
print("✓")

# Feature 2: How much interest in products vs information
print("   - ProductInfoRatio (are they focused on products?)", end=" ")
df["ProductInfoRatio"] = df["ProductRelated"] / (df["Informational"] + 1)
print("✓")

# Feature 3: How valuable was the time spent?
print("   - EngagementScore (value gained per minute)", end=" ")
df["EngagementScore"] = df["PageValues"] / (df["TotalSessionDuration"] + 1)
print("✓")

# 4.5) Create CYCLIC features for months (because Dec→Jan is close, not opposite)
print("\n5️⃣ Creating seasonal (Month_sin & Month_cos) features...", end=" ")
df["Month_sin"] = np.sin(2 * np.pi * df["Month_num"] / 12.0)  # Sine representation
df["Month_cos"] = np.cos(2 * np.pi * df["Month_num"] / 12.0)  # Cosine representation
# Remove temporary numeric month and the original text `Month` column so numeric pipeline doesn't see strings
df = df.drop(columns=["Month_num", "Month"])
print("Done!")

# 4.6) Final checks: Make sure target is clean
print("\n6️⃣ Final validation...", end=" ")
if df[TARGET_COL].isna().any():
    raise ValueError("❌ Revenue has invalid values after cleaning!")
df[TARGET_COL] = df[TARGET_COL].astype(int)
df["Weekend"] = df["Weekend"].astype(int)
print("Done!")

print("\n" + "="*60)
print(f"✓ Cleaned data shape: {df.shape}")
print(f"✓ New features created: 5 (TotalSessionDuration, ProductInfoRatio,")
print(f"                        EngagementScore, Month_sin, Month_cos)")
print(f"✓ Total rows kept: {len(df)} (duplicates NOT removed)")
print("="*60)

df.head()



CLEANING DATA...

1️⃣ Checking for duplicate rows... Found 125 duplicate rows
   ℹ️  Note: These represent real visitor sessions with identical patterns.
   They are KEPT (not removed) because each row = one session.

2️⃣ Converting Weekend & Revenue to 0/1 format... Done!
3️⃣ Converting month names to numbers... Done!

4️⃣ Creating engineered features...
   - TotalSessionDuration (sum of all page durations) ✓
   - ProductInfoRatio (are they focused on products?) ✓
   - EngagementScore (value gained per minute) ✓

5️⃣ Creating seasonal (Month_sin & Month_cos) features... Done!

6️⃣ Final validation... Done!

✓ Cleaned data shape: (12330, 22)
✓ New features created: 5 (TotalSessionDuration, ProductInfoRatio,
                        EngagementScore, Month_sin, Month_cos)
✓ Total rows kept: 12330 (duplicates NOT removed)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Region,TrafficType,VisitorType,Weekend,Revenue,TotalSessionDuration,ProductInfoRatio,EngagementScore,Month_sin,Month_cos
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,1,1,Returning_Visitor,0,0,0.000000,1.0,0.0,0.866025,0.5
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,...,1,2,Returning_Visitor,0,0,64.000000,2.0,0.0,0.866025,0.5
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,...,9,3,Returning_Visitor,0,0,0.000000,1.0,0.0,0.866025,0.5
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,...,2,4,Returning_Visitor,0,0,2.666667,2.0,0.0,0.866025,0.5
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,...,1,4,Returning_Visitor,1,0,627.500000,10.0,0.0,0.866025,0.5


In [42]:
# ======================================================================
# STEP 5: SEPARATE FEATURES & BUILD PREPROCESSING PIPELINE
# ======================================================================
# A 'pipeline' is a chain of transformations to apply to our data

# Separate target (what we predict) from features (what we use to predict)
X = df.drop(columns=[TARGET_COL])  # Features (everything except Revenue)
y = df[TARGET_COL]  # Target (Revenue = 0 or 1)

# Identify which columns are categorical vs numerical
# We already created Month_sin/Month_cos (cyclic). Keep original `Month` out of
# categorical features to avoid duplication — the numeric cyclic columns remain
# in `numerical_features`.
categorical_features = ["VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
numerical_features = [col for col in X.columns if col not in categorical_features]

print("="*60)
print("STEP 5: BUILD PREPROCESSING PIPELINE")
print("="*60)
print(f"\n📊 Features split:")
print(f"   • Numerical ({len(numerical_features)}): {numerical_features}")
print(f"   • Categorical ({len(categorical_features)}): {categorical_features}")

# PIPELINE FOR NUMERICAL FEATURES
print(f"\n📐 Numerical Pipeline:")
print(f"   1. Fill missing values with MEDIAN")
print(f"   2. Standardize (scale to mean=0, std=1)")
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# PIPELINE FOR CATEGORICAL FEATURES
print(f"\n🏷️  Categorical Pipeline:")
print(f"   1. Fill missing values with MOST COMMON value")
print(f"   2. One-Hot Encode (convert text to 0/1 columns)")
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# COMBINE BOTH PIPELINES
print(f"\n🔗 Combining pipelines...")
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

print(f"   ✓ Done!")
print("="*60)

STEP 5: BUILD PREPROCESSING PIPELINE

📊 Features split:
   • Numerical (16): ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Weekend', 'TotalSessionDuration', 'ProductInfoRatio', 'EngagementScore', 'Month_sin', 'Month_cos']
   • Categorical (5): ['VisitorType', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']

📐 Numerical Pipeline:
   1. Fill missing values with MEDIAN
   2. Standardize (scale to mean=0, std=1)

🏷️  Categorical Pipeline:
   1. Fill missing values with MOST COMMON value
   2. One-Hot Encode (convert text to 0/1 columns)

🔗 Combining pipelines...
   ✓ Done!


In [43]:
# ======================================================================
# STEP 6: APPLY PIPELINE TO TRANSFORM DATA
# ======================================================================

print("\n" + "="*60)
print("TRANSFORMING DATA...")
print("="*60)

# Apply the pipeline - this actually does the transformations
print("\n⚙️  Applying transformations...", end=" ")
X_processed = preprocessor.fit_transform(X)
print("Done!")

# Get the names of all output columns after encoding
# Make this robust to different scikit-learn versions / pipeline shapes
cat_feature_names = []
cat_transformer = preprocessor.named_transformers_.get("cat")
# If categorical pipeline is a Pipeline, extract the encoder step
if hasattr(cat_transformer, "named_steps"):
    encoder = cat_transformer.named_steps.get("encoder")
else:
    encoder = cat_transformer

if encoder is None:
    cat_feature_names = np.array([])
else:
    try:
        cat_feature_names = encoder.get_feature_names_out(categorical_features)
    except Exception:
        try:
            cat_feature_names = encoder.get_feature_names(categorical_features)
        except Exception:
            # Fallback: build names from encoder.categories_ if present
            if hasattr(encoder, "categories_"):
                names = []
                for feat, cats in zip(categorical_features, encoder.categories_):
                    names.extend([f"{feat}__{str(c)}" for c in cats])
                cat_feature_names = np.array(names)
            else:
                cat_feature_names = np.array([])

processed_feature_names = list(numerical_features) + list(cat_feature_names)

# Convert to a DataFrame for easy viewing and saving
X_processed_df = pd.DataFrame(X_processed, columns=processed_feature_names, index=X.index)

print(f"✓ Transformation complete!")
print(f"   Input shape: {X.shape[0]} rows × {X.shape[1]} features")
print(f"   Output shape: {X_processed_df.shape[0]} rows × {X_processed_df.shape[1]} features")

print("\n" + "="*60)
print("CHECKING FOR DATA DRIFT")
print("="*60)

# Check if data has changed over time (important for monitoring in production)
# Split data into early 70% (reference) vs late 30% (current)
print("\n📈 Computing Population Stability Index (PSI)...")
print("   (Measures if feature distributions have changed)\n")

split_idx = int(len(df) * 0.7)
reference_df = df.iloc[:split_idx]  # First 70% = baseline
current_df = df.iloc[split_idx:]   # Last 30% = recent

# Function to calculate PSI
def calculate_psi(reference_series: pd.Series, current_series: pd.Series, bins: int = 10) -> float:
    """Calculate how much a feature's distribution has changed."""
    # Clean: remove infinities and missing values
    reference_series = reference_series.replace([np.inf, -np.inf], np.nan).dropna()
    current_series = current_series.replace([np.inf, -np.inf], np.nan).dropna()
    if reference_series.empty or current_series.empty:
        return 0.0

    # Create bins and count values in each bin
    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(np.quantile(reference_series, quantiles))
    if len(edges) < 3:
        return 0.0

    ref_counts, _ = np.histogram(reference_series, bins=edges)
    cur_counts, _ = np.histogram(current_series, bins=edges)

    # Convert to percentages
    ref_ratio = np.clip(ref_counts / max(ref_counts.sum(), 1), 1e-6, 1)
    cur_ratio = np.clip(cur_counts / max(cur_counts.sum(), 1), 1e-6, 1)

    # PSI formula: sum((current% - reference%) × ln(current% / reference%))
    psi = float(np.sum((cur_ratio - ref_ratio) * np.log(cur_ratio / ref_ratio)))
    return psi

# Calculate PSI for each numerical feature
psi_records = []
for col in numerical_features:
    psi_value = calculate_psi(reference_df[col], current_df[col])
    drift_flag = psi_value > 0.2  # Flag if drift is significant
    psi_records.append({
        "feature": col,
        "psi": round(psi_value, 4),
        "drift_flag": drift_flag,
    })


drift_report = pd.DataFrame(psi_records).sort_values("psi", ascending=False)

print("Top 10 Features with Most Drift:")
print(drift_report.head(10).to_string(index=False))

print("\n" + "="*60)



TRANSFORMING DATA...

⚙️  Applying transformations... Done!
✓ Transformation complete!
   Input shape: 12330 rows × 21 features
   Output shape: 12330 rows × 69 features

CHECKING FOR DATA DRIFT

📈 Computing Population Stability Index (PSI)...
   (Measures if feature distributions have changed)

Top 10 Features with Most Drift:
                feature     psi  drift_flag
              Month_cos 12.6993        True
              Month_sin 11.6717        True
             SpecialDay  1.4748        True
         ProductRelated  0.0930       False
ProductRelated_Duration  0.0919       False
   TotalSessionDuration  0.0904       False
       ProductInfoRatio  0.0554       False
              ExitRates  0.0525       False
            BounceRates  0.0400       False
          Informational  0.0153       False



In [44]:
# ======================================================================
# STEP 7: SAVE PROCESSED DATA & METADATA
# ======================================================================

print("\n" + "="*60)
print("SAVING ARTIFACTS")
print("="*60)

# Save the preprocessor object so we can use it on new data later
print("\n💾 Saving files...")
print(f"   1. preprocessor.joblib - For transforming new data")
joblib.dump(preprocessor, DATA_ARTIFACT_DIR / "preprocessor.joblib")

# Save the processed features
print(f"   2. X_processed.csv - Transformed features")
X_processed_df.to_csv(DATA_ARTIFACT_DIR / "X_processed.csv", index=False)

# Save the target variable
print(f"   3. y.csv - Target variable (0=no purchase, 1=purchased)")
y.to_csv(DATA_ARTIFACT_DIR / "y.csv", index=False)

# Save the drift report
print(f"   4. drift_report.csv - Data drift monitoring")
drift_report.to_csv(DATA_ARTIFACT_DIR / "drift_report.csv", index=False)

# Create a summary of what we did
num_high_drift = int((drift_report["drift_flag"] == True).sum())
metadata = {
    "data_version": DATA_VERSION,
    "description": "Online Shoppers - Preprocessed & Ready for ML",
    "total_rows": len(df),
    "total_features_before_encoding": len(X.columns),
    "total_features_after_encoding": X_processed_df.shape[1],
    "target_name": TARGET_COL,
    "positive_rate": round(float(y.mean() * 100), 2),
    "new_engineered_features": ["TotalSessionDuration", "ProductInfoRatio", "EngagementScore", "Month_sin", "Month_cos"],
    "features_with_high_drift": num_high_drift,
}

# Save metadata as JSON
print(f"   5. metadata.json - Summary information")
with open(DATA_ARTIFACT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\n" + "="*60)
print("✅ DATA PIPELINE COMPLETE!")
print("="*60)
print(f"\n📁 All files saved to: {DATA_ARTIFACT_DIR}")
print(f"\n📋 Summary:")
print(f"   • Total rows: {metadata['total_rows']}")
print(f"   • Positive rate: {metadata['positive_rate']}% (purchased)")
print(f"   • Features created: {len(metadata['new_engineered_features'])}")
print(f"   • Data drift detected: {num_high_drift} features")
print("\n✓ Ready for ML model training!")
print("="*60)


SAVING ARTIFACTS

💾 Saving files...
   1. preprocessor.joblib - For transforming new data
   2. X_processed.csv - Transformed features
   3. y.csv - Target variable (0=no purchase, 1=purchased)
   4. drift_report.csv - Data drift monitoring
   5. metadata.json - Summary information

✅ DATA PIPELINE COMPLETE!

📁 All files saved to: artifacts\v2

📋 Summary:
   • Total rows: 12330
   • Positive rate: 15.47% (purchased)
   • Features created: 5
   • Data drift detected: 3 features

✓ Ready for ML model training!
